# 🚀 BOT TRADE — Entraînement TFT sur GPU (H100)

Notebook pour entraîner le **Temporal Fusion Transformer** sur BTC/USDT depuis Binance.

### Etapes :
1. Verification GPU
2. Upload du projet sur Google Drive
3. Installation des dependances
4. Telechargement des donnees historiques Binance
5. Entrainement TFT full-size (hidden=256, 14M params)
6. Entrainement RL (PPO)
7. Sauvegarde des checkpoints vers Google Drive

---
**Runtime recommande :** `Runtime > Change runtime type > GPU > H100`

## Etape 1 — Verification du GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'PyTorch version  : {torch.__version__}')
print(f'CUDA disponible  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU              : {torch.cuda.get_device_name(0)}')
    print(f'VRAM             : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Pas de GPU detecte -> Runtime > Change runtime type > GPU')


## Etape 2 — Google Drive

**Avant de lancer cette cellule :**
1. Zippe ton dossier `BOT_TRADE` en `BOT_TRADE.zip`
2. Upload `BOT_TRADE.zip` dans **Mon Drive/** (racine de ton Google Drive)
3. Lance la cellule ci-dessous


In [ ]:
from google.colab import drive
import os, zipfile, shutil

drive.mount('/content/drive')

ZIP_PATH    = '/content/drive/MyDrive/BOT_TRADE.zip'
EXTRACT_PATH = '/content/BOT_TRADE'

if os.path.exists(EXTRACT_PATH):
    shutil.rmtree(EXTRACT_PATH)
    print('Ancien dossier supprime.')

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall('/content/')
print(f'Projet extrait dans {EXTRACT_PATH}')

os.chdir(EXTRACT_PATH)
print(f'Repertoire courant : {os.getcwd()}')
print('Fichiers :', os.listdir('.'))


## Etape 3 — Installation des dependances

In [ ]:
import subprocess, sys

packages = [
    'ccxt==4.3.80',
    'pandas>=2.1.0',
    'numpy>=1.26.0',
    'pandas-ta>=0.4.0',
    'pyarrow>=14.0.0',
    'pytorch-forecasting>=1.0.0',
    'stable-baselines3>=2.2.0',
    'gymnasium>=0.29.0',
    'sb3-contrib>=2.2.0',
    'shimmy>=1.3.0',
    'vectorbt>=0.25.5',
    'optuna>=3.4.0',
    'plotly>=5.18.0',
    'tensorboard>=2.15.0',
    'rich>=13.7.0',
    'tqdm>=4.66.0',
    'pyyaml>=6.0.1',
    'python-dotenv>=1.0.0',
    'sqlalchemy>=2.0.25',
    'aiofiles>=23.2.1',
    'cryptography>=41.0.7',
]

print('Installation des packages...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + packages,
    capture_output=True, text=True
)
if result.returncode != 0:
    print('ERREUR:', result.stderr[-3000:])
else:
    print('Tous les packages installes avec succes')


In [ ]:
import torch, pytorch_forecasting, stable_baselines3, vectorbt
print(f'torch               : {torch.__version__}')
print(f'pytorch-forecasting : {pytorch_forecasting.__version__}')
print(f'stable-baselines3   : {stable_baselines3.__version__}')
print(f'vectorbt            : {vectorbt.__version__}')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'
print(f'CUDA                : {torch.cuda.is_available()} — {gpu_name}')


## Etape 4 — Configuration GPU

On remet les hyperparametres full-size (hidden=256, 14M params) maintenant qu'on a un H100.


In [ ]:
import yaml, os
os.chdir('/content/BOT_TRADE')

CONFIG_PATH = 'config/config.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Full-size TFT pour H100
cfg['models']['tft']['hidden_size']         = 256
cfg['models']['tft']['lstm_layers']         = 2
cfg['models']['tft']['num_attention_heads'] = 8
cfg['models']['tft']['horizon']             = 12
cfg['models']['tft']['batch_size']          = 256
cfg['models']['tft']['max_epochs']          = 100
cfg['models']['tft']['learning_rate']       = 1e-3

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)

PAIRS = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT']

print('Config mise a jour pour GPU :')
for k, v in cfg['models']['tft'].items():
    print(f'   {k}: {v}')


## Etape 5 — Telechargement des donnees historiques (Binance mainnet, public)

In [ ]:
import subprocess, sys

for pair in PAIRS:
    print(f'Telechargement {pair}...')
    result = subprocess.run(
        [sys.executable, 'main.py', '--mode', 'download',
         '--pairs', pair, '--start', '2022-01-01'],
        capture_output=True, text=True, cwd='/content/BOT_TRADE'
    )
    lines = (result.stdout + result.stderr).strip().split('\n')
    for line in lines[-8:]:
        print(line)
    status = 'OK' if result.returncode == 0 else 'ERREUR'
    print(f'  -> {status}\n')

print('Donnees telecharges.')


## Etape 6 — Entrainement TFT

Avec un H100 et 14M params :
- batch_size=256, ~109 steps/epoch
- Environ **3-5 min par epoch**
- 100 epochs avec early stopping -> **~3-8h** selon convergence

Lance TensorBoard dans la cellule suivante pour monitorer en direct.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/BOT_TRADE/logs/tensorboard


In [ ]:
import sys, os
sys.path.insert(0, '/content/BOT_TRADE')
os.chdir('/content/BOT_TRADE')

import torch
from data.collectors.historical import HistoricalDataCollector
from data.processors.features import FeatureEngineer
from models.tft_model import TFTPredictor
from utils.logger import setup_logger

logger = setup_logger('colab_train')

device = 'GPU -- ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Device : {device}')

collector = HistoricalDataCollector()
fe        = FeatureEngineer()
tft       = TFTPredictor()

trained_models = {}

for pair in PAIRS:
    print('=' * 60)
    print(f'  Entrainement TFT -- {pair}')
    print('=' * 60)
    try:
        df = collector.load_data(pair, '1h')
        if df is None or len(df) < 500:
            print(f'Pas assez de donnees pour {pair}, skip.')
            continue
        print(f'{len(df)} bougies chargees')

        features = fe.compute_features(df)
        print(f'{features.shape[1]} features, {len(features)} lignes')

        model = tft.train(features, pair)
        trained_models[pair] = model
        print(f'TFT entraine pour {pair}')

    except Exception as e:
        print(f'Erreur {pair} : {e}')
        import traceback; traceback.print_exc()

print(f'Entrainement termine : {list(trained_models.keys())}')


## Etape 7 — Entrainement RL (PPO)

In [ ]:
from models.rl_agent import RLTradingAgent

rl_agents = {}

for pair in PAIRS:
    print('=' * 60)
    print(f'  Entrainement RL -- {pair}')
    print('=' * 60)
    try:
        df       = collector.load_data(pair, '1h')
        features = fe.compute_features(df)

        agent = RLTradingAgent()
        agent.train(features, symbol=pair)
        rl_agents[pair] = agent
        print(f'RL entraine pour {pair}')

    except Exception as e:
        print(f'Erreur {pair} : {e}')
        import traceback; traceback.print_exc()

print(f'RL termine : {list(rl_agents.keys())}')


## Etape 8 — Courbes d'entrainement

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import matplotlib.pyplot as plt
import glob

fig, axes = plt.subplots(1, len(PAIRS), figsize=(6 * len(PAIRS), 4))
if len(PAIRS) == 1:
    axes = [axes]

for ax, pair in zip(axes, PAIRS):
    safe = pair.replace('/', '_')
    dirs = glob.glob(f'/content/BOT_TRADE/logs/tensorboard/tft_{safe}*')
    if not dirs:
        ax.set_title(f'{pair} -- pas de logs')
        continue
    ea = EventAccumulator(sorted(dirs)[-1])
    ea.Reload()
    for tag in ['train_loss', 'val_loss']:
        if tag in ea.Tags().get('scalars', []):
            evts = ea.Scalars(tag)
            ax.plot([e.step for e in evts], [e.value for e in evts], label=tag)
    ax.set_title(f'TFT -- {pair}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('QuantileLoss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
out = '/content/BOT_TRADE/logs/training_curves.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Courbes sauvegardees : {out}')


## Etape 9 — Sauvegarde des checkpoints vers Google Drive

In [ ]:
import shutil, os
from datetime import datetime

ts = datetime.now().strftime('%Y%m%d_%H%M')
dst = f'/content/drive/MyDrive/BOT_TRADE_checkpoints_{ts}'
os.makedirs(dst, exist_ok=True)

for sub in ['tft', 'rl']:
    src = f'/content/BOT_TRADE/checkpoints/{sub}'
    if os.path.exists(src):
        shutil.copytree(src, f'{dst}/{sub}', dirs_exist_ok=True)
        n = len(os.listdir(f'{dst}/{sub}'))
        print(f'  {n} fichiers {sub.upper()} sauvegardes')

curves = '/content/BOT_TRADE/logs/training_curves.png'
if os.path.exists(curves):
    shutil.copy(curves, dst)
    print('  Courbes sauvegardees')

print(f'Tout sauvegarde dans : {dst}')
print('-> Copie checkpoints/tft/ et checkpoints/rl/ dans BOT_TRADE/ sur ton PC')


---
## Recuperation des modeles sur ton PC

1. Telecharge le dossier `BOT_TRADE_checkpoints_YYYYMMDD_HHMM` depuis Google Drive
2. Copie `tft/*.ckpt` et `tft/*.pkl` dans `BOT_TRADE/checkpoints/tft/`
3. Copie `rl/*.zip` dans `BOT_TRADE/checkpoints/rl/`
4. Lance le live trading :
   ```bash
   python main.py --mode live --pairs BTC/USDT ETH/USDT SOL/USDT
   ```
